# pamqp

## Example 1: the protocol header

In [ ]:
# Example 1: the 8-byte AMQP 0-9-1 protocol header a client sends first
from pamqp import frame, header

raw = frame.marshal(header.ProtocolHeader(), 0)
assert raw == b"AMQP\x00\x00\x09\x01", raw
consumed, channel, value = frame.unmarshal(raw)
assert consumed == 8 and isinstance(value, header.ProtocolHeader)
print("Example 1:", raw, (value.major_version, value.minor_version, value.revision))

## Example 2: a method frame round trip

In [ ]:
# Example 2: Queue.Declare marshals to a method frame and unmarshals to an equal command on the same channel
from pamqp import commands, frame

declare = commands.Queue.Declare(queue="jobs", durable=True, arguments={"x-max-priority": 10})
raw = frame.marshal(declare, 7)
consumed, channel, decoded = frame.unmarshal(raw)
assert consumed == len(raw) and channel == 7
assert isinstance(decoded, commands.Queue.Declare)
assert (decoded.queue, decoded.durable, decoded.arguments) == ("jobs", True, {"x-max-priority": 10})
assert raw[-1:] == b"\xce"  # every AMQP frame ends with the 0xCE frame-end octet
print("Example 2:", decoded.name, decoded.queue, decoded.arguments, len(raw), "bytes")

## Example 3: a content header carrying message properties

In [ ]:
# Example 3: Basic.Properties travel in a content header; the decoded header keeps size and every property
from pamqp import commands, frame, header

props = commands.Basic.Properties(content_type="application/json", delivery_mode=2, headers={"retry": 3})
raw = frame.marshal(header.ContentHeader(0, 42, props), 1)
_, channel, decoded = frame.unmarshal(raw)
assert isinstance(decoded, header.ContentHeader) and decoded.body_size == 42
assert decoded.properties.content_type == "application/json"
assert (decoded.properties.delivery_mode, decoded.properties.headers) == (2, {"retry": 3})
print("Example 3:", decoded.body_size, decoded.properties.content_type, decoded.properties.headers)

## Example 4: field-table encoding and decoding

In [ ]:
# Example 4: encode.field_table / decode.field_table round-trip AMQP's typed table values
import datetime
from pamqp import decode, encode

table = {"int": 1, "text": "hi", "flag": True, "nested": {"x": 1.5}, "list": [1, "a"],
         "when": datetime.datetime(2026, 1, 2, 3, 4, 5, tzinfo=datetime.timezone.utc)}
raw = encode.field_table(table)
consumed, value = decode.field_table(raw)
assert consumed == len(raw)
assert value == table, value
print("Example 4:", len(raw), "bytes ->", sorted(value))

## Example 5: a truncated frame is detected

In [ ]:
# Example 5: unmarshal refuses a frame cut short, the case a network reader must handle
from pamqp import commands, exceptions, frame

raw = frame.marshal(commands.Basic.Publish(exchange="ex", routing_key="rk"), 1)
for cut in (len(raw) - 1, 5):
    try:
        frame.unmarshal(raw[:cut])
    except exceptions.UnmarshalingException as error:
        print(f"Example 5: {cut} of {len(raw)} bytes refused: {type(error).__name__}")
    else:
        raise AssertionError(f"a frame cut to {cut} bytes was accepted")